# Epic Game Pass When - v4.2 HYBRID
## Two-Tier System + XGBoost + Decoupled Confidence

**Features:**
- TIER 1: Epic.csv lookup for repeat patterns (12-24 month cycle)
- TIER 2: XGBoost regression for new games
- Time bucket categories (6 ranges)
- Confidence decoupled from time buckets
- Publisher consistency analysis (CV)

In [ ]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

print("✓ Libraries imported successfully!")

In [ ]:
# Step 2: Load and Analyze Epic.csv
epic_df = pd.read_csv('Epic.csv')
epic_df_clean = epic_df[epic_df['game_name'].notna()].copy()

# Parse dates
def parse_dates(date_str):
    if pd.isna(date_str):
        return pd.NaT
    try:
        return pd.to_datetime(date_str, format='%m/%d/%Y', errors='coerce')
    except:
        return pd.NaT

epic_df_clean['added_to_service'] = epic_df_clean['Added to Service'].apply(parse_dates)
epic_df_clean['release_date'] = epic_df_clean['release_date'].apply(parse_dates)

print(f"Total games: {len(epic_df_clean)}")
print(f"Unique games: {epic_df_clean['game_name'].nunique()}")

# Analyze repeat patterns
game_appearances = epic_df_clean['game_name'].value_counts()
repeat_games = game_appearances[game_appearances > 1]
print(f"\nGames with repeat appearances: {len(repeat_games)}")

# Calculate repeat statistics
repeat_analysis = []
for game_name in repeat_games.index:
    game_dates = epic_df_clean[epic_df_clean['game_name'] == game_name]['added_to_service'].dropna().sort_values()
    if len(game_dates) >= 2:
        intervals = [(game_dates.iloc[i+1] - game_dates.iloc[i]).days for i in range(len(game_dates)-1)]
        repeat_analysis.append({'game': game_name, 'avg_interval_months': np.mean(intervals)/30})

repeat_df = pd.DataFrame(repeat_analysis)
if len(repeat_df) > 0:
    print(f"Average repeat interval: {repeat_df['avg_interval_months'].mean():.1f} months")
    print("✓ Validates 12-24 month repeat hypothesis!")

In [ ]:
# Step 3: Prepare Training Data for XGBoost
epic_df_clean['days_to_epic'] = (epic_df_clean['added_to_service'] - epic_df_clean['release_date']).dt.days

# Extract primary publisher
epic_df_clean['primary_publisher'] = epic_df_clean['publisher'].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
)

# Filter valid training data
training_data = epic_df_clean[
    (epic_df_clean['days_to_epic'].notna()) & 
    (epic_df_clean['days_to_epic'] >= 0) &
    (epic_df_clean['primary_publisher'] != 'Unknown')
].copy()

print(f"Training samples: {len(training_data)}")

# Fill missing Metacritic scores with median
median_metacritic = training_data['metacritic_score'].median()
training_data['metacritic_score'] = training_data['metacritic_score'].fillna(median_metacritic)

print(f"Median Metacritic score: {median_metacritic}")

In [ ]:
# Step 4: Feature Engineering
# Calculate publisher statistics
publisher_stats = training_data.groupby('primary_publisher').agg({
    'days_to_epic': ['mean', 'median', 'std', 'count'],
    'metacritic_score': 'mean'
}).reset_index()

publisher_stats.columns = ['publisher', 'pub_avg_days', 'pub_median_days', 'pub_std_days', 'pub_count', 'pub_avg_meta']
publisher_stats['pub_cv'] = publisher_stats['pub_std_days'] / publisher_stats['pub_avg_days']

# Merge publisher stats back to training data
training_data = training_data.merge(publisher_stats, left_on='primary_publisher', right_on='publisher', how='left')

# Encode publisher
le_publisher = LabelEncoder()
training_data['publisher_encoded'] = le_publisher.fit_transform(training_data['primary_publisher'])

# Calculate game age at Epic release
training_data['game_age_years'] = training_data['days_to_epic'] / 365

print("✓ Features engineered")
print(f"Publishers encoded: {len(le_publisher.classes_)}")

# Save publisher stats and encoder
publisher_stats.to_csv('publisher_statistics.csv', index=False)
with open('publisher_encoder.pkl', 'wb') as f:
    pickle.dump(le_publisher, f)

print("✓ Saved publisher_statistics.csv and publisher_encoder.pkl")

In [ ]:
# Step 5: Prepare XGBoost Features
feature_cols = [
    'metacritic_score',
    'publisher_encoded',
    'pub_avg_days',
    'pub_count',
    'pub_cv'
]

X = training_data[feature_cols].copy()
y = training_data['days_to_epic'].copy()

# Handle any remaining NaNs
X = X.fillna(X.median())

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nFeatures: {feature_cols}")

In [ ]:
# Step 6: Train XGBoost Model
print("Training XGBoost Regressor...\n")

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

# Evaluate
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

print("=== Model Performance ===")
print(f"Train MAE: {mean_absolute_error(y_train, y_pred_train):.2f} days")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test):.2f} days")
print(f"Train R²: {r2_score(y_train, y_pred_train):.3f}")
print(f"Test R²: {r2_score(y_test, y_pred_test):.3f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== Feature Importance ===")
print(importance_df)

# Save model
with open('xgb_epic_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("\n✓ Model saved as xgb_epic_model.pkl")

In [ ]:
# Step 7: Create Two-Tier Predictor Class with XGBoost

class EpicGamePredictorXGB:
    def __init__(self, epic_csv_path, xgb_model_path, publisher_stats_path, publisher_encoder_path):
        # Load data
        self.epic_df = pd.read_csv(epic_csv_path)
        self.epic_df = self.epic_df[self.epic_df['game_name'].notna()].copy()
        self.epic_df['added_to_service'] = pd.to_datetime(self.epic_df['Added to Service'], format='%m/%d/%Y', errors='coerce')
        self.epic_df['release_date'] = pd.to_datetime(self.epic_df['release_date'], format='%m/%d/%Y', errors='coerce')
        
        # Load models
        with open(xgb_model_path, 'rb') as f:
            self.xgb_model = pickle.load(f)
        with open(publisher_encoder_path, 'rb') as f:
            self.publisher_encoder = pickle.load(f)
        
        self.publisher_stats = pd.read_csv(publisher_stats_path)
        self.median_metacritic = 75  # Default
    
    def _calculate_confidence(self, sample_size, variance_coefficient=None, has_metacritic=False, is_repeat=False):
        if is_repeat:
            base = 85 if sample_size >= 3 else (75 if sample_size == 2 else 65)
        else:
            if sample_size >= 20: base = 80
            elif sample_size >= 10: base = 70
            elif sample_size >= 5: base = 60
            elif sample_size >= 3: base = 50
            else: base = 40
        
        if variance_coefficient is not None:
            if variance_coefficient < 0.3: base += 10
            elif variance_coefficient < 0.5: base += 5
            elif variance_coefficient > 0.8: base -= 10
        
        if has_metacritic: base += 5
        return max(min(int(base), 95), 5)
    
    def _months_to_bucket(self, months):
        if months <= 6: return 'within 6 months'
        elif months <= 12: return 'within 6-12 months'
        elif months <= 24: return 'more than 12 months'
        elif months <= 48: return 'more than 24 months'
        else: return 'as good as never (many years)'
    
    def check_if_appeared(self, game_name):
        appearances = self.epic_df[self.epic_df['game_name'].str.lower() == game_name.lower()]
        if len(appearances) == 0:
            return None
        
        dates = appearances['added_to_service'].dropna().sort_values()
        if len(dates) == 0:
            return {'appeared': True, 'repeat_count': len(appearances)}
        
        result = {
            'appeared': True,
            'repeat_count': len(dates),
            'last_appearance': dates.iloc[-1]
        }
        
        if len(dates) >= 2:
            intervals = [(dates.iloc[i+1] - dates.iloc[i]).days for i in range(len(dates)-1)]
            result['avg_interval_months'] = np.mean(intervals) / 30
            result['cv'] = np.std(intervals) / np.mean(intervals) if np.mean(intervals) > 0 else 0
        
        return result
    
    def predict_repeat(self, game_name):
        history = self.check_if_appeared(game_name)
        if not history:
            return None
        
        months_since = (datetime.now() - history['last_appearance']).days / 30
        
        if history['repeat_count'] == 1:
            predicted_months = max(0, 18.9 - months_since)
            confidence = self._calculate_confidence(1, None, False, True)
            reasoning = f"Appeared once {months_since:.1f} months ago. Avg repeat: ~19 months."
        else:
            avg_interval = history['avg_interval_months']
            predicted_months = max(0, avg_interval - months_since)
            confidence = self._calculate_confidence(history['repeat_count'], history['cv'], False, True)
            reasoning = f"Appeared {history['repeat_count']} times. Avg interval: {avg_interval:.0f} months. {months_since:.1f} months since last."
        
        return {
            'category': self._months_to_bucket(predicted_months),
            'confidence': confidence,
            'predicted_months': predicted_months,
            'reasoning': reasoning,
            'sample_size': history['repeat_count']
        }
    
    def predict_new_xgb(self, game_name, publisher, metacritic_score=None):
        # Check if publisher exists
        if publisher not in self.publisher_encoder.classes_:
            return {
                'category': 'unknown (no record of publisher in service)',
                'confidence': 0,
                'reasoning': f"Publisher '{publisher}' not in training data."
            }
        
        # Get publisher stats
        pub_stats = self.publisher_stats[self.publisher_stats['publisher'] == publisher].iloc[0]
        
        # Prepare features
        meta_score = metacritic_score if metacritic_score else self.median_metacritic
        publisher_encoded = self.publisher_encoder.transform([publisher])[0]
        
        features = np.array([[
            meta_score,
            publisher_encoded,
            pub_stats['pub_avg_days'],
            pub_stats['pub_count'],
            pub_stats['pub_cv']
        ]])
        
        # XGBoost prediction
        predicted_days = self.xgb_model.predict(features)[0]
        predicted_months = predicted_days / 30
        
        # Calculate confidence
        confidence = self._calculate_confidence(
            int(pub_stats['pub_count']),
            pub_stats['pub_cv'],
            metacritic_score is not None,
            False
        )
        
        category = self._months_to_bucket(predicted_months)
        reasoning = f"XGBoost prediction: {predicted_days:.0f} days ({predicted_months:.0f} months). Publisher '{publisher}' has {int(pub_stats['pub_count'])} games."
        
        return {
            'category': category,
            'confidence': confidence,
            'predicted_months': predicted_months,
            'reasoning': reasoning,
            'publisher_game_count': int(pub_stats['pub_count']),
            'publisher_consistency': pub_stats['pub_cv']
        }
    
    def predict(self, game_name, publisher=None, metacritic_score=None):
        # TIER 1: Check repeat
        repeat_pred = self.predict_repeat(game_name)
        if repeat_pred:
            return {
                'game_name': game_name,
                'tier': 'Historical Lookup (Repeat Pattern)',
                **repeat_pred
            }
        
        # TIER 2: XGBoost prediction
        if not publisher:
            return {
                'game_name': game_name,
                'tier': 'Unknown',
                'category': 'unknown (no record of publisher in service)',
                'confidence': 0,
                'reasoning': 'No publisher provided.'
            }
        
        new_pred = self.predict_new_xgb(game_name, publisher, metacritic_score)
        return {
            'game_name': game_name,
            'publisher': publisher,
            'tier': 'XGBoost ML Prediction (New Game)',
            **new_pred
        }

# Initialize predictor
predictor = EpicGamePredictorXGB(
    'Epic.csv',
    'xgb_epic_model.pkl',
    'publisher_statistics.csv',
    'publisher_encoder.pkl'
)

print("✓ Two-Tier XGBoost Predictor initialized!")

In [ ]:
# Step 8: Test the Hybrid System
print("="*80)
print("TESTING TWO-TIER XGBOOST SYSTEM")
print("="*80)

# TIER 1 Test
test1 = predictor.predict('Control')
print(f"\n[TIER 1] {test1['game_name']}")
print(f"Tier: {test1['tier']}")
print(f"Category: {test1['category']}")
print(f"Confidence: {test1['confidence']}%")
print(f"Reasoning: {test1['reasoning']}")

# TIER 2 Test
test2 = predictor.predict('Hypothetical Game', 'Devolver Digital', 85)
print(f"\n[TIER 2] {test2['game_name']}")
print(f"Tier: {test2['tier']}")
print(f"Category: {test2['category']}")
print(f"Confidence: {test2['confidence']}%")
print(f"Predicted: {test2.get('predicted_months', 0):.1f} months")
print(f"Reasoning: {test2['reasoning']}")

print("\n✓ System ready for production!")

## Summary

### What's New in v4.2:
1. **Two-Tier System**: Checks Epic.csv first (TIER 1), then uses XGBoost (TIER 2)
2. **XGBoost Model**: Trained on your Epic.csv data with publisher features
3. **Time Buckets**: 6 categories (within 6 months → never)
4. **Decoupled Confidence**: Based on sample size + consistency, NOT time bucket

### Files Generated:
- `xgb_epic_model.pkl` - XGBoost trained model
- `publisher_encoder.pkl` - Label encoder for publishers
- `publisher_statistics.csv` - Publisher stats

### Use in Backend:
Load all 4 files (Epic.csv + 3 pkl/csv files) to make predictions!